# Initializing the environment

In [12]:
os.path.exists("results")

False

In [ ]:
from gymnasium import make
from samples.llm_interface import SoloGPT4Interfacer, MultiGPT4Interfacer
import os
import random
from natural20.gym.tools import compute_available_moves
os.environ["OPENAI_API_KEY"] = ""

n_player1 = 2
n_player2 = 2
# Initialize the environment
# env = make("dndenv-v0", root_path="templates", render_mode="ansi")
all_classes = ['halfling_rogue.yml', 'high_elf_fighter.yml']
all_players = ["Alysha", "Bernard", "Cedric", "Didier", "Eric", "Francois", "Gertrude", "Heloise", "Isabelle"]
players = random.sample(all_players, n_player1 + n_player2)
a_player = [(random.choice(all_classes), player) for player in players[:n_player1]]
e_player = [(random.choice(all_classes), player) for player in players[n_player1:]]

conversational_groups = {"a":True, "b":False}
env = make(
    "dndenv-v0",
    render_mode="ansi",
    map_file="maps/game_map.yml",
    show_logs=True,
    profiles=a_player,
    enemies=e_player,
    control_groups=["a","b"]
    )

envi = env.env.env
observation, info = env.reset(seed = random.randint(0,1000))




agents = {}
groups = env.env.env.battle.groups  
for character in env.env.env.battle.combat_order:
    gr = None
    for group_name, group in env.env.env.battle.groups.items():
        if character in group:
            gr = group_name
    if gr == None : print(f"Warning ! : Character {character.name} has no friends !!! (aka no groups attributed)")
    agent_type = MultiGPT4Interfacer if conversational_groups[gr] else SoloGPT4Interfacer
    agents[character.name] = (agent_type(debug=False, explain=True, name=character.name), gr, character)

agents


Francois rolled initiative d20(20) + 5 value 25.2
Alysha rolled initiative d20(11) + 5 value 16.2
Eric rolled initiative d20(4) + 5 value 9.2
Bernard rolled initiative d20(10) + 5 value 15.2
Francois rolled initiative d20(15) + 5 value 20.2
Alysha rolled initiative d20(5) + 5 value 10.2
Eric rolled initiative d20(10) + 5 value 15.2
Bernard rolled initiative d20(9) + 5 value 14.2
Combat begins with 4 players.
Players: <p>Francois (fighter-2) Team a</p>
<p>Alysha (rogue-2) Team a</p>
<p>Eric (fighter-2) Team b</p>
<p>Bernard (fighter-2) Team b</p>
======== Francois starts their turn. ========
======== Francois starts their turn. ========


/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:130: UserWarning: WARN: The obs returned by the `reset()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/spaces/box.py:423: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


{'Francois': (<samples.llm_interface.MultiGPT4Interfacer at 0x7fa59174f9d0>,
  'a',
  Francois),
 'Eric': (<samples.llm_interface.SoloGPT4Interfacer at 0x7fa59174fc50>,
  'b',
  Eric),
 'Bernard': (<samples.llm_interface.SoloGPT4Interfacer at 0x7fa4dcd94d60>,
  'b',
  Bernard),
 'Alysha': (<samples.llm_interface.MultiGPT4Interfacer at 0x7fa4dcd94e90>,
  'a',
  Alysha)}

In [25]:
class Experiment():
    def __init__(self, environment, dnd_environment, agents, conversational_groups, debug=False):
        self.env = environment
        self.dnd_environment = dnd_environment
        self.agents = agents
        self.debug = debug
        self.conversational_groups = conversational_groups
        self.backlog = []
        self.conversations = []
        self.finished = False
        self.current_step = 0
    
    def get_obs_inf(self, player):
        p_observation = self.dnd_environment.generate_observation(player)
        p_available_moves = compute_available_moves(self.dnd_environment.session, self.dnd_environment.map, player, self.dnd_environment.battle, self.dnd_environment.weapon_mappings, self.dnd_environment.spell_mappings)
        p_info = self.dnd_environment._info(p_available_moves, player)
        return p_observation, p_info
    
    def update_all_agents(self, agents_name, sender, content):
        for name in agents_name:
            self.agents[name][0].register_conversation(sender, content)
        self.conversations[-1].append((sender, content))

    def initiate_conversation(self, agents_name):
        for name in agents_name:
            self.agents[name][0].initiate_conversation()
        self.conversations.append([])

    def close_conversation(self, agents_name):
        for name in agents_name:
            obs, inf = self.get_obs_inf(self.agents[name][2])
            self.agents[name][0].close_conversation(obs, inf, self.get_players_pos())
        self.conversations[-1].append((None, "Conversation closed"))

    def run_conversation(self, sender, content):
        sender_gr = self.agents[sender][1]
        if not self.conversational_groups[sender_gr]:
            raise ValueError(f"The agent {sender} from the non conversational group {sender_gr} tried to initiate a conversation")
        agent_in_the_conv = []
        for name, (_, gr, _) in self.agents.items():
            if sender_gr == gr and name != sender:
                agent_in_the_conv.append(name)
        agent_in_the_conv.append(sender)
        self.initiate_conversation(agent_in_the_conv)
        self.update_all_agents(agent_in_the_conv, sender, content)
        conv_alive = True
        conv_step = 0
        while conv_alive:
            conv_step += 1
            conv_alive = False
            for name in agent_in_the_conv:
                obs, inf = self.get_obs_inf(self.agents[name][2])
                action, descrition,  content = self.agents[name][0].select_action_for_state(obs, inf, self.get_players_pos(), is_conversation=True)
                if action == -2:
                    self.update_all_agents(agent_in_the_conv, name, content)
                    conv_alive = True
                elif action != -3:
                    raise ValueError(f"A non conversation action {action} was used during a conversation by agent {name}")
        self.close_conversation(agent_in_the_conv)
    
    def step(self):
        current_agent, current_group, current_character = self.agents[self.dnd_environment.battle.current_turn().name]
        obs, inf = self.get_obs_inf(current_character)
        # Manual removala of help action since it crashes
        help_index = []
        for i, el in enumerate(inf["available_moves"]):
            if el[0] == 14:
                help_index.append(i)
        for index in help_index[::-1]:
            inf["available_moves"].pop(index)
        
        action, descrition, content = current_agent.select_action_for_state(obs, inf, self.get_players_pos())
        if self.debug:
            print(f"The chosen action is : {action}")
        self.backlog.append((current_character.name, action, descrition))
        if action != -1:
            _, _, terminal, _, _ = self.env.step(action)
        else :
            self.backlog.append((current_character.name, -1, len(self.conversations)))
            self.run_conversation(sender=current_character.name, content=content)
            terminal = False
        return terminal
    
    def get_players_pos(self):
        return {player: self.dnd_environment.battle.maps[0].entity_or_object_pos(player) for (_, _, player) in self.agents.values()}
    
    def run_till_end(self, max_step= 30):
        done = False
        while not done and self.current_step < max_step:
            if self.debug:
                print(f"\n\n________________________________________________________________________________\n Starting step {self.current_step}:\n")
                health = []
                for player, pos in self.get_players_pos().items():
                    health.append(player.health_percent())
                    print(f"Player {player.name} has {player.health_percent()}% life points, {pos}")
            done = self.step()
            self.current_step += 1
        self.finished = self.dnd_environment.battle.battle_ends() or (self.current_step >= max_step)
        return self.finished

In [27]:
expe = Experiment(env, env.env.env, agents, conversational_groups=conversational_groups, debug=False)
expe.run_till_end(max_step=100)

{'action_id': 0, 'description': 'end my turn', 'explanation': 'I have no available actions left and have already used my movement this turn. Ending my turn is the only option.'}
Alysha makes a death saving throw and succeeds: d20(11) = 11
======== Alysha starts their turn. ========
{'action_id': 0, 'description': 'end my turn', 'explanation': "I am at 0 HP and incapacitated, so I cannot take any actions except for possibly making death saving throws, which should be handled by the DM's process. I must end my turn."}
======== Francois starts their turn. ========
======== Francois starts their turn. ========
{'action_id': 6, 'description': 'dodge action', 'explanation': "I'm critically wounded and facing two enemies, one of whom is dodging, and the other at full health. My priority remains survival. By taking the Dodge action, I maximize my chances of avoiding incoming attacks this round and buy time to look for a safer opening."}
Francois dodges.
{'action_id': 0, 'description': 'End my 

True

In [28]:
from natural20.gym.llm_helpers.metrics import combat_metrics, combat_score


expe.backlog
metrics = combat_metrics(expe.dnd_environment)
print(metrics)
score = combat_score(metrics)
print(score)

{'win': False, 'turns_taken': 115, 'survivors': {'Francois': (0, 24), 'Alysha': (0, 16), 'Eric': (5, 24), 'Bernard': (24, 24)}}
-102.91666666666667


In [29]:
metrics

{'win': False,
 'turns_taken': 115,
 'survivors': {'Francois': (0, 24),
  'Alysha': (0, 16),
  'Eric': (5, 24),
  'Bernard': (24, 24)}}

In [24]:
expe.conversations

[[('Alysha',
   "Francois, I'm badly hurt and staying hidden for now. Focus on keeping Eric and Bernard busy—I'll look for a moment to strike safely. Try to draw their attention if you can!"),
  ('Francois',
   "Understood, Alysha! I'll keep the pressure on Eric and try to draw both enemies' attention. Stay safe—strike whenever you see an opening. If they split up, I'll call it out so we can take advantage. Let's keep each other posted."),
  (None, 'Conversation closed')],
 [('Francois',
   "Alysha, Eric is badly hurt but is Dodging right now, making him harder to hit. Stay safe and ready—we'll finish him as soon as an opportunity opens up. Once Eric is down, we both shift our focus to Bernard. Let me know if you need support or see a better move."),
  ('Alysha',
   'Thanks, Francois. I’m keeping my head down for now—no openings yet, but I’m watching for any slip in Eric’s defense. If he drops his guard, I’ll move in for a Sneak Attack. If things get dicey or you need a distraction, gi